# 05 — LSTM sketch (**incomplete**)

This notebook was never executed. It assumes `X_train`, `y_train`, and `device` already exist in a kernel that previously ran `04_inverse_nn.ipynb`. It is kept only as a record of a sequential-model idea; it is not a result.


### **Step 1: Retain Preprocessing Steps**
No changes are needed here. Your existing preprocessing pipeline (normalization, feature extraction, scaling) works for both SimpleNN and LSTM.



### **Step 2: Reshape Data for LSTM**
Add this as a new cell after the preprocessing section.

In [ ]:
import torch

# Function to reshape data for LSTM (expects 3D input: batch_size, seq_length, num_features)
def reshape_for_lstm(X, sequence_length):
    """
    Reshape X into 3D shape for LSTM: (batch_size, sequence_length, num_features)
    """
    num_samples = X.shape[0] - sequence_length + 1
    X_lstm = []

    for i in range(num_samples):
        X_lstm.append(X[i:i+sequence_length].values)

    return torch.FloatTensor(X_lstm)

# Set sequence length (e.g., 24 time steps for a full day if data is hourly)
sequence_length = 24

# Reshape the training and validation data
X_train_lstm = reshape_for_lstm(X_train, sequence_length)
y_train_lstm = y_train[sequence_length-1:].values  # Align target data
X_val_lstm = reshape_for_lstm(X_val, sequence_length)
y_val_lstm = y_val[sequence_length-1:].values

# Convert target to tensor
y_train_lstm = torch.FloatTensor(y_train_lstm)
y_val_lstm = torch.FloatTensor(y_val_lstm)

print(f"LSTM Train Data Shape: {X_train_lstm.shape}")
print(f"LSTM Validation Data Shape: {X_val_lstm.shape}")



## **Step 3: Define LSTM Model**
Add this cell after the existing SimpleNN definition.

In [ ]:
import torch.nn as nn

class LSTMNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2, dropout=0.2):
        super(LSTMNN, self).__init__()

        # LSTM layer
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout)

        # Fully connected layer for prediction
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # LSTM forward pass
        lstm_out, _ = self.lstm(x)
        
        # Use the output of the last time step for prediction
        out = self.fc(lstm_out[:, -1, :])
        return out

# Hyperparameters
input_size = X_train_lstm.shape[2]    # Number of input features
hidden_size = 128                     # LSTM hidden units
output_size = y_train_lstm.shape[1]   # 3 targets: capacity, tilt, orientation
num_layers = 2                        # Number of LSTM layers

# Initialize model
lstm_model = LSTMNN(input_size, hidden_size, output_size, num_layers).to(device)


## **Step 4: Train LSTM Model**
Add a new cell after your existing training loop for SimpleNN.

In [ ]:
import torch.optim as optim

# Loss and optimizer
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Training loop
epochs = 50
for epoch in range(epochs):
    lstm_model.train()
    optimizer.zero_grad()

    # Forward pass
    outputs = lstm_model(X_train_lstm.to(device))
    loss = criterion(outputs, y_train_lstm.to(device))

    # Backward pass and optimization
    loss.backward()
    optimizer.step()

    # Validation step
    lstm_model.eval()
    with torch.no_grad():
        val_outputs = lstm_model(X_val_lstm.to(device))
        val_loss = criterion(val_outputs, y_val_lstm.to(device))

    # Print progress
    print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}')


## **Step 5: Evaluate and Compare with SimpleNN**
Add a new cell to visualize and compare predictions.

In [ ]:
import matplotlib.pyplot as plt

# Make predictions
lstm_model.eval()
with torch.no_grad():
    lstm_predictions = lstm_model(X_val_lstm.to(device)).cpu().numpy()

# Plot true vs predicted for each target
target_names = ['pv_dc_kwp', 'pv_array_tilt', 'pv_array_a_orientation']

for i, target in enumerate(target_names):
    plt.figure(figsize=(10, 4))
    plt.plot(y_val_lstm[:, i], label='True', alpha=0.7)
    plt.plot(lstm_predictions[:, i], label='LSTM Predicted', alpha=0.7)
    plt.title(f'{target} Prediction - LSTM')
    plt.legend()
    plt.show()
